# Setup

In [ ]:
import os
import time
import subprocess
import requests

print("Setting up Docker + Qdrant for Kaggle...")

# 1) Install Docker runtime (quiet mode for notebook logs).
subprocess.run("apt-get update -qq", shell=True, check=False)
subprocess.run("apt-get install -y -qq docker.io", shell=True, check=False)

# 2) Start Docker daemon (Kaggle often has no systemd).
subprocess.run("pkill -f dockerd", shell=True, check=False)
subprocess.Popen(["dockerd"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

docker_ready = False
for _ in range(30):
    info = subprocess.run("docker info", shell=True, capture_output=True, text=True)
    if info.returncode == 0:
        docker_ready = True
        break
    time.sleep(1)

if not docker_ready:
    raise RuntimeError("Docker daemon did not start. Please restart the notebook runtime and try again.")

print("Docker is ready")

# 3) Run Qdrant container and persist data under /kaggle/working.
os.makedirs("/kaggle/working/qdrant_storage", exist_ok=True)
subprocess.run("docker rm -f qdrant-local", shell=True, check=False)
run_cmd = (
    "docker run -d --name qdrant-local "
    "-p 6333:6333 "
    "-v /kaggle/working/qdrant_storage:/qdrant/storage "
    "qdrant/qdrant:latest"
)
started = subprocess.run(run_cmd, shell=True, capture_output=True, text=True)
if started.returncode != 0:
    raise RuntimeError(f"Failed to start Qdrant container: {started.stderr.strip()}")

# 4) Healthcheck Qdrant API.
qdrant_ready = False
for _ in range(30):
    try:
        r = requests.get("http://localhost:6333/collections", timeout=2)
        if r.status_code == 200:
            qdrant_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not qdrant_ready:
    raise RuntimeError("Qdrant is not responding on http://localhost:6333")

print("Qdrant is ready at http://localhost:6333")

In [ ]:
import subprocess

# Install GPU detection tools
subprocess.run("apt-get install -qq pciutils lshw", shell=True)
print("✅ pciutils and lshw installed")

# Check if Kaggle's GPU is visible
result = subprocess.run("lspci | grep -i nvidia", shell=True, capture_output=True, text=True)
print("GPU detected:", result.stdout.strip())

# Check nvidia driver
result2 = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
print(result2.stdout[:500])

In [ ]:
import subprocess
import time
import requests

# ── 1. Install zstd (required by Ollama installer) ──────────────────
print("Installing zstd...")
subprocess.run("apt-get install -qq zstd", shell=True)
print("✅ zstd installed")

# ── 2. Install Ollama ────────────────────────────────────────────────
print("\nInstalling Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
print("✅ Ollama installed")

# ── 3. Verify binary exists ──────────────────────────────────────────
which = subprocess.run("which ollama", shell=True, capture_output=True, text=True)
print(f"\nOllama binary: {which.stdout.strip()}")

# ── 4. Start Ollama server ───────────────────────────────────────────
print("\nStarting Ollama server...")
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

# ── 5. Verify server is up ───────────────────────────────────────────
try:
    response = requests.get("http://localhost:11434/api/tags")
    print("✅ Ollama server is running")
except Exception as e:
    print(f"❌ Server not responding: {e}")

# ── 6. Pull models ───────────────────────────────────────────────────
models = [
    "qwen2.5:7b-instruct", 
    "llama3.2-vision:11b",
    "nomic-embed-text"
]

for model in models:
    print(f"\nPulling {model}...")
    subprocess.run(["ollama", "pull", model])
    print(f"✅ {model} ready")

# ── 7. Final check ───────────────────────────────────────────────────
response = requests.get("http://localhost:11434/api/tags")
available = [m["name"] for m in response.json()["models"]]
print(f"\n✅ All done! Models available: {available}")

In [ ]:
!pip install -q \
    unstructured[all-docs] \
    langchain \
    langchain-community \
    langchain-ollama \
    langchain-chroma \
    chromadb \
    pillow \
    pdf2image \
    pdfminer.six \
    python-dotenv \
    psutil \
    

# unstructured needs these system packages for PDF parsing
!apt-get install -qq poppler-utils tesseract-ocr libmagic1

In [ ]:
# Test all critical imports
packages = {
    "unstructured": "from unstructured.partition.pdf import partition_pdf",
    "langchain": "from langchain_core.documents import Document",
    "langchain_ollama": "from langchain_ollama import ChatOllama, OllamaEmbeddings",
    "langchain_chroma": "from langchain_chroma import Chroma",
    "chromadb": "import chromadb",
}

all_good = True
for name, import_stmt in packages.items():
    try:
        exec(import_stmt)
        print(f"✅ {name}")
    except ImportError as e:
        print(f"❌ {name}: {e}")
        all_good = False

print("\n✅ All imports OK — proceed!" if all_good else "\n❌ Fix the failed imports above first")

In [ ]:
import os

os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"
os.environ["TEXT_MODEL"] = "qwen2.5:7b-instruct"
os.environ["VISION_MODEL"] = "llama3.2-vision:11b"
os.environ["EMBED_MODEL"] = "nomic-embed-text"
os.environ["ENABLE_VISION"] = "true"

In [ ]:
import subprocess, time, requests, os

subprocess.run("pkill -f ollama", shell=True)
time.sleep(3)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1"

subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

requests.get("http://localhost:11434/api/tags")
print("✅ Ollama ready with both GPUs")

# Begin

In [1]:
import json
from typing import List
import gc
import psutil
import time 
import subprocess
import requests

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.partition.auto import partition
from unstructured.chunking.title import chunk_by_title

# Langchain components
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

from langchain_ollama import ChatOllama, OllamaEmbeddings

# Try to import torch for GPU cleanup (optional)
try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False

load_dotenv(override=True)

# Local Ollama configuration (optimized for your laptop profile)
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
TEXT_MODEL = os.getenv("TEXT_MODEL", "qwen2.5:7b-instruct")
VISION_MODEL = os.getenv("VISION_MODEL", "llama3.2-vision:11b")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")
ENABLE_VISION = os.getenv("ENABLE_VISION", "true").lower() == "true"

# Shared local embedding model for Chroma
embedding_model = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)

/home/luminous/miniconda3/envs/env_ml/lib/python3.12/site-packages/torch/cuda/__init__.py:184: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
print(f"Configuration: TEXT_MODEL={TEXT_MODEL}, VISION_MODEL={VISION_MODEL}, EMBEDDING_MODEL={EMBED_MODEL}, ENABLE_VISION={ENABLE_VISION}")

Configuration: TEXT_MODEL=qwen2.5:0.5b, VISION_MODEL=moondream:latest, EMBEDDING_MODEL=nomic-embed-text, ENABLE_VISION=True


In [22]:
def partition_document(file_path: str):
    """Extract elements from supported document formats (.pdf, .csv, .docx, .doc, .txt)."""
    print(f"Partitioning document: {file_path}")

    extension = os.path.splitext(file_path)[1].lower()
    supported_extensions = {".pdf", ".csv", ".docx", ".doc", ".txt"}
    
    if extension not in supported_extensions:
        raise ValueError(
            f"Unsupported file type: {extension}. "
            f"Supported types: {', '.join(sorted(supported_extensions))}"
        )

    if extension == ".pdf":
        elements = partition_pdf(
            filename=file_path,                     # Path to the PDF file
            strategy="hi_res",                      # Most accurate but slower for layout-aware extraction
            infer_table_structure=True,             # Keep tables as structured HTML
            extract_image_block_type=["Image"],     # Extract images for multimodal summary
            extract_image_block_to_payload=True,    # Keep image payload as base64
            extract_images_in_pdf=True
        )
    else:
        # Generic parser path for csv/docx/doc/txt
        elements = partition(filename=file_path)

    print(f"Extracted {len(elements)} elements from the document.")
    return elements

In [23]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy."""
    print("Creating chunks based on titles...")

    chunks = chunk_by_title(
        elements=elements,                  # The list of elements extracted from the PDF
        max_characters=3000,                # Maximum number of characters in each chunk - Hard limit
        new_after_n_chars=2400,             # Soft limit - if a new title is encountered after this many characters, start a new chunk anyway
        combine_text_under_n_chars=500,     # If a title is followed by less than this many characters, combine it with the next chunk instead of starting a new one
    )

    print(f"Created {len(chunks)} chunks based on titles.")
    return chunks

In [24]:
def seperate_content_types(chunk):
    """Separate text, tables, and images from a chunk."""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    # Check for tables and images in original chunk elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)

            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)

    content_data['types'] = list(set(content_data['types']))
    return content_data


def _build_summary_prompt(text, tables, image_count):
    """Create a retrieval-oriented prompt for summary generation."""
    prompt_text = f"""You are creating a searchable description for document retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        YOUR TASK:
        Generate a comprehensive, searchable description that covers:
        1. Key facts, numbers, and data points from text and tables
        2. Main topics and concepts discussed
        3. Questions this content could answer
        4. Alternative search terms users might use
        5. If visual information exists, include likely chart/diagram intent

        Prioritize findability over brevity.
        """

    if tables:
        prompt_text += "\n\nTABLES:\n"
        for i, table in enumerate(tables, start=1):
            prompt_text += f"Table {i}:\n{table}\n\n"

    if image_count > 0:
        prompt_text += f"\nThis chunk contains {image_count} image(s). Include visual context when possible.\n"

    prompt_text += "\nSEARCHABLE DESCRIPTION:"
    return prompt_text


def _estimate_base64_size_mb(image_base64: str) -> float:
    """Approximate decoded image size in MB from base64 length."""
    if not image_base64:
        return 0.0
    return (len(image_base64) * 3 / 4) / (1024 * 1024)


def _prepare_images_for_vision(images, max_images=1, max_image_mb=1.5, max_total_mb=2.0):
    """Filter large/invalid images to reduce vision model crashes."""
    selected_images = []
    total_mb = 0.0

    for image_base64 in images:
        if not image_base64 or not isinstance(image_base64, str):
            continue

        img_mb = _estimate_base64_size_mb(image_base64)
        if img_mb > max_image_mb:
            print(f"Skipping one image (~{img_mb:.2f}MB): exceeds max_image_mb={max_image_mb}")
            continue

        if total_mb + img_mb > max_total_mb:
            print(f"Stopping image add: total image payload would exceed {max_total_mb}MB")
            break

        selected_images.append(image_base64)
        total_mb += img_mb

        if len(selected_images) >= max_images:
            break

    return selected_images


def _invoke_summary_model(model_name, prompt_text, images_base64, num_ctx, timeout):
    llm = ChatOllama(
        model=model_name,
        temperature=0,
        base_url=OLLAMA_BASE_URL,
        num_ctx=num_ctx,
        timeout=timeout,
        keep_alive=0,
    )
    message_content = [{"type": "text", "text": prompt_text}]
    for image_base64 in images_base64:
        message_content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"},
        })
    message = HumanMessage(content=message_content)
    response = llm.invoke([message])
    return response.content

In [25]:
# ===== GPU/Ollama watchdog =====
GPU_RESTART_THRESHOLD = 0.92
GPU_RESTART_COOLDOWN_SEC = 60
OLLAMA_HEALTHCHECK_RETRIES = 20
OLLAMA_HEALTHCHECK_SLEEP_SEC = 1.0
_last_ollama_restart_at = 0.0


def _get_gpu_memory_ratios():
    cmd = [
        "nvidia-smi",
        "--query-gpu=index,memory.used,memory.total",
        "--format=csv,noheader,nounits",
    ]
    res = subprocess.run(cmd, capture_output=True, text=True, check=False)
    if res.returncode != 0 or not res.stdout.strip():
        return []

    rows = []
    for line in res.stdout.strip().splitlines():
        i, used, total = [x.strip() for x in line.split(",")]
        used = float(used)
        total = max(float(total), 1.0)
        rows.append((int(i), used, total, used / total))
    return rows


def _restart_ollama_server():
    global _last_ollama_restart_at

    user = os.getenv("USER", "")
    if user:
        subprocess.run(["pkill", "-u", user, "-f", "ollama serve"], check=False)
    else:
        subprocess.run(["pkill", "-f", "ollama serve"], check=False)
    time.sleep(2)

    env = os.environ.copy()
    env.setdefault("OLLAMA_NUM_PARALLEL", "1")
    env.setdefault("OLLAMA_MAX_LOADED_MODELS", "1")

    subprocess.Popen(
        ["ollama", "serve"],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    ok = False
    for _ in range(OLLAMA_HEALTHCHECK_RETRIES):
        try:
            r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
            if r.status_code == 200:
                ok = True
                break
        except Exception:
            pass
        time.sleep(OLLAMA_HEALTHCHECK_SLEEP_SEC)

    _last_ollama_restart_at = time.time()
    if ok:
        print("Ollama restarted and healthy")
    else:
        print("Ollama restarted but healthcheck not ready yet")


def _maybe_restart_ollama_on_gpu_pressure(threshold=GPU_RESTART_THRESHOLD):
    global _last_ollama_restart_at

    if time.time() - _last_ollama_restart_at < GPU_RESTART_COOLDOWN_SEC:
        return False

    stats = _get_gpu_memory_ratios()
    if not stats:
        return False

    idx, used, total, ratio = max(stats, key=lambda x: x[3])
    if ratio >= threshold:
        print(f"GPU {idx} high usage: {used:.0f}/{total:.0f} MB ({ratio:.1%}) -> restarting Ollama")
        _restart_ollama_server()
        return True
    return False

In [26]:
def create_ai_summary(text, tables, images):
    """Vision retry once after Ollama restart, then fallback to text."""
    _maybe_restart_ollama_on_gpu_pressure()

    prompt_text = _build_summary_prompt(text, tables, len(images))
    safe_images = _prepare_images_for_vision(
        images,
        max_images=1,
        max_image_mb=1.0,
        max_total_mb=1.0,
    )

    use_vision = ENABLE_VISION and len(safe_images) > 0

    # 1) Vision path (first attempt)
    if use_vision:
        try:
            content = _invoke_summary_model(
                model_name=VISION_MODEL,
                prompt_text=prompt_text,
                images_base64=safe_images,
                num_ctx=2048,
                timeout=600,
            )
            return {
                "summary": content,
                "used_fallback": False,
                "model_used": VISION_MODEL,
                "images_sent": len(safe_images),
            }
        except Exception as e1:
            print(f"Vision attempt #1 failed: {e1}")
            print("Restarting Ollama and retrying vision once...")
            _restart_ollama_server()

            # 2) Vision retry after restart
            try:
                content = _invoke_summary_model(
                    model_name=VISION_MODEL,
                    prompt_text=prompt_text,
                    images_base64=safe_images,
                    num_ctx=2048,
                    timeout=600,
                )
                return {
                    "summary": content,
                    "used_fallback": False,
                    "model_used": VISION_MODEL,
                    "images_sent": len(safe_images),
                }
            except Exception as e2:
                print(f"Vision attempt #2 failed: {e2}")
                print("Falling back to text model...")

    # 3) Text fallback
    try:
        text_only_prompt = _build_summary_prompt(text, tables, image_count=0)
        content = _invoke_summary_model(
            model_name=TEXT_MODEL,
            prompt_text=text_only_prompt,
            images_base64=[],
            num_ctx=3072,
            timeout=300,
        )
        return {
            "summary": content,
            "used_fallback": True,
            "model_used": TEXT_MODEL,
            "images_sent": 0,
        }
    except Exception as text_exc:
        print(f"Text fallback failed: {text_exc}")

    # 4) Local final fallback
    summary = f"{text[:300]}..."
    if tables:
        summary += f" [Contains {len(tables)} table(s)]"
    if images:
        summary += f" [Contains {len(images)} image(s)]"
    return {
        "summary": summary,
        "used_fallback": True,
        "model_used": "local-fallback",
        "images_sent": 0,
    }


def summarize_chunks(chunks, source_file):
    """Process all chunks with AI summarization + GPU watchdog."""
    print("Processing chunks with AI summarization...")
    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        _maybe_restart_ollama_on_gpu_pressure()

        print(f"Summarizing chunk {current_chunk}/{total_chunks}...")
        content_data = seperate_content_types(chunk)

        print(f"Types found: {content_data['types']}")
        print(f"Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}, Text chars: {len(content_data['text'])}")

        if content_data['tables'] or content_data['images']:
            print("Chunk contains tables/images, creating AI-enhanced summary...")
            result = create_ai_summary(
                content_data['text'],
                content_data['tables'],
                content_data['images'],
            )
            summary = result["summary"]
            print(f"Summary path: model={result['model_used']}, fallback={result['used_fallback']}, images_sent={result['images_sent']}")
            print(f"Summary: {summary[:200]}...")
        else:
            print("Chunk does not contain tables/images, using original text as summary.")
            summary = content_data["text"]

        doc = Document(
            page_content=summary,
            metadata={
                "source_file": source_file,
                "original_content": json.dumps({
                    "raw_text": content_data["text"],
                    "tables_html": content_data["tables"],
                    "images_base64": content_data["images"],
                })
            },
        )
        langchain_documents.append(doc)

    print(f"\nSummarized {len(langchain_documents)} chunks into Langchain Documents.")
    return langchain_documents

In [27]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"Exported {len(export_data)} chunks to {filename}")
    return export_data

In [34]:
INGEST_TRACKING_FILE = "ingested_files.json"


def load_ingested_files(tracking_file=INGEST_TRACKING_FILE):
    """Load ingested filenames from local tracker."""
    if not os.path.exists(tracking_file):
        return set()
    try:
        with open(tracking_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                return set(data)
    except Exception as e:
        print(f"Warning: cannot read {tracking_file}: {e}")
    return set()


def save_ingested_file(source_file, tracking_file=INGEST_TRACKING_FILE):
    """Persist a newly ingested filename to local tracker."""
    files = load_ingested_files(tracking_file)
    files.add(source_file)
    with open(tracking_file, "w", encoding="utf-8") as f:
        json.dump(sorted(files), f, indent=2, ensure_ascii=False)


def is_file_already_ingested(source_file, tracking_file=INGEST_TRACKING_FILE):
    """Check whether a file (by basename) was previously ingested."""
    return source_file in load_ingested_files(tracking_file)


def _ensure_collection_exists(client, embedding_model, collection_name):
    """Create collection if it does not exist (needed after reset)."""
    try:
        exists = client.collection_exists(collection_name=collection_name)
    except Exception:
        try:
            client.get_collection(collection_name=collection_name)
            exists = True
        except Exception:
            exists = False

    if exists:
        return

    vector_dim = len(embedding_model.embed_query("dimension_probe"))
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE),
    )
    print(f"Created collection: {collection_name} (dim={vector_dim})")


def create_vector_store(
    embedding_model,
    documents,
    collection_name="rag_docs",
    qdrant_url="http://localhost:6333"
):
    print("Creating embeddings and storing in Qdrant...")

    client = QdrantClient(url=qdrant_url, timeout=10)
    _ensure_collection_exists(client, embedding_model, collection_name)

    vectorstore = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embedding_model,
    )

    if documents:
        vectorstore.add_documents(documents)
        print(f"Added {len(documents)} documents to collection: {collection_name}")
    else:
        print("No new documents to add.")

    return vectorstore


def reset_all_database(
    collection_name="dbv2_docs",
    qdrant_url="http://localhost:6333",
    tracking_file=INGEST_TRACKING_FILE,
):
    """Delete all vectors in collection and reset local ingest tracker."""
    client = QdrantClient(url=qdrant_url, timeout=10)

    try:
        client.delete_collection(collection_name=collection_name)
        print(f"Deleted collection: {collection_name}")
    except Exception as e:
        print(f"Collection delete skipped/failed: {e}")

    with open(tracking_file, "w", encoding="utf-8") as f:
        json.dump([], f, indent=2)
    print(f"Reset tracking file: {tracking_file}")

    global db
    db = None
    print("Database reset complete.")

In [35]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("Starting RAG Ingestion Pipeline")
    print("=" * 50)

    source_file = os.path.basename(pdf_path)
    if is_file_already_ingested(source_file):
        print(f"Skipping ingestion: '{source_file}' already exists in tracking list.")
        return create_vector_store(
            embedding_model,
            documents=[],
            collection_name="dbv2_docs",
            qdrant_url="http://localhost:6333",
        )

    # Step 1: Partition
    elements = partition_document(pdf_path)

    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)

    # Step 3: AI Summarisation
    summarised_chunks = summarize_chunks(chunks, source_file=source_file)

    # Step 4: Vector Store
    db = create_vector_store(
        embedding_model,
        summarised_chunks,
        collection_name="dbv2_docs",
        qdrant_url="http://localhost:6333",
    )

    save_ingested_file(source_file)
    # export_chunks_to_json(summarised_chunks, filename="new_version.json")

    print("Pipeline completed successfully!")
    return db

In [37]:
def generate_final_answer(query):
    """Generate final answer using multimodal content"""
    
    retriever = db.as_retriever(search_kwargs={"k": 5})
    chunks = retriever.invoke(query)
    export_chunks_to_json(chunks, filename="retrieved_chunks_new_version.json")
    try:
        # Initialize LLM (needs vision model for images)
        # llm = ChatOpenAI(model="gpt-4o", temperature=0)
        llm = ChatOllama(
            model=TEXT_MODEL,
            temperature=0,
            base_url="http://localhost:11434",
        )
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}
                    CONTENT TO ANALYZE:
                    """
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
                Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

                ANSWER:
                """

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # # This is for model like ChatGPT or Gemini
        # # Add all images from all chunks
        # for chunk in chunks:
        #     if "original_content" in chunk.metadata:
        #         original_data = json.loads(chunk.metadata["original_content"])
        #         images_base64 = original_data.get("images_base64", [])
                
        #         for image_base64 in images_base64:
        #             message_content.append({
        #                 "type": "image_url",
        #                 "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
        #             })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

In [39]:
db = run_complete_ingestion_pipeline("./docs/Nvidia.txt")

Starting RAG Ingestion Pipeline
Partitioning document: ./docs/Nvidia.txt
Extracted 459 elements from the document.
Creating chunks based on titles...
Created 62 chunks based on titles.
Processing chunks with AI summarization...
Summarizing chunk 1/62...
Types found: ['text']
Tables: 0, Images: 0, Text chars: 989
Chunk does not contain tables/images, using original text as summary.
Summarizing chunk 2/62...
Types found: ['text']
Tables: 0, Images: 0, Text chars: 1784
Chunk does not contain tables/images, using original text as summary.
Summarizing chunk 3/62...
Types found: ['text']
Tables: 0, Images: 0, Text chars: 2922
Chunk does not contain tables/images, using original text as summary.
Summarizing chunk 4/62...
Types found: ['text']
Tables: 0, Images: 0, Text chars: 2807
Chunk does not contain tables/images, using original text as summary.
Summarizing chunk 5/62...
Types found: ['text']
Tables: 0, Images: 0, Text chars: 2414
Chunk does not contain tables/images, using original text 

In [40]:
generate_final_answer("when was Google found?")

Exported 5 chunks to retrieved_chunks_new_version.json


'To determine when Google was founded, I\'ll analyze the given documents step by step:\n\n1. **Document 1**: This document provides a brief history of Google\'s development and founding.\n   - It mentions that Google began in January 1996 as a research project by Larry Page and Sergey Brin while they were both PhD students at Stanford University.\n\n2. **Document 2**:\n   - This document lists several key figures involved in the development of Google, including Peter Beal, Susan Wojcicki, Vangie Beal, Elinor Mills, Rajeev Motwani, and Terry Winograd.\n   - It states that these individuals were "Credited to Page and Brin as being critical to the development of Google."\n\n3. **Document 4**:\n   - This document provides information about Google\'s initial funding: an August 1998 investment of $100,000 from Andy Bechtolsheim.\n\nBased on this analysis:\n\n- The first document (Document 1) clearly states that Google was founded in January 1996.\n- Document 2 confirms that Peter Beal and Su

In [41]:
generate_final_answer("when is Nvidia Corporation headquartered?")

Exported 5 chunks to retrieved_chunks_new_version.json


'Based on the information provided in the documents:\n\n1. **Google\'s Headquarters Location**: The text states that Google\'s headquarters is located in Santa Clara, California.\n\n2. **Nvidia Corporation\'s Headquarters Location**: The text mentions that Nvidia Corporation\'s headquarters are in Santa Clara, California.\n\n3. **Timeline of Document 5**:\n   - In 2006, Google moved into about 300,000 square feet (27,900 m²) of office space at 111 Eighth Avenue in Manhattan.\n   - In 2018, Google announced its plan to expand its New York City office to a capacity of 12,000 employees.\n\n4. **Timeline of Document 5**:\n   - In 2006, Google established a new headquarters for its AdWords division in Ann Arbor, Michigan.\n   - In November 2006, Google opened offices on Carnegie Mellon\'s campus in Pittsburgh, focusing on shopping-related advertisement coding and smartphone applications and programs.\n\nTherefore, the answer to the question "When is Nvidia Corporation headquartered?" based 

In [42]:
generate_final_answer("when was SpaceX founded?")

Exported 5 chunks to retrieved_chunks_new_version.json


'To determine when SpaceX was founded, I\'ll analyze the given documents step by step:\n\n1. **Document 1**: This document mentions that in 2004, Google formed the not-for-profit philanthropic organization "Google.org" with a start-up fund of $1 billion.\n   - From this information, we can infer that SpaceX was founded in 2004.\n\n2. **Document 5**: This document states that in 2003, after outgrowing two other locations, Google leased an office complex from Silicon Graphics at 1600 Amphitheatre Parkway in Mountain View, California.\n   - From this information, we can infer that SpaceX was founded in 2003.\n\n3. **Document 4**: This document mentions that in 2007, Google acquired Junglee, which was the founding of the company\'s "not the usual" philanthropic foundation.\n   - From this information, we can infer that SpaceX was founded in 2007.\n\n4. **Document 5**: This document states that in 2011, Google donated €1 million to the IMO (International Mathematical Olympiad).\n   - From t

In [31]:
reset_all_database()

Deleted collection: dbv2_docs
Reset tracking file: ingested_files.json
Database reset complete.
